
# Swapping the nebular backend on, then off, on a young starburst

Hα and [O III]+Hβ are produced by gas reprocessing the ionizing
continuum from O/B stars. The SFH is a young starburst (peak age ≈ 30 Myr).

Reference: Li+2025 (Cue emulator; arXiv:2405.04598).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore")

ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")

# A young starburst: peak ~30 Myr ago, still ionizing
sfh = {
    "type": "dpl",
    "all_params": tengri.FIXED,
    "alpha": 3.0,
    "beta": 0.3,
    "tau_gyr": 0.03,
    "log_total_mass": 10.0,
}
dust = {
    "law": "power_law",
    "type": "two_component",
    "all_params": tengri.FIXED,
    "tau_diff": 0.05,
    "tau_bc": 0.0,
}

model_cue = tengri.SEDModel.build(
    ssp,
    sfh=sfh,
    dust=dust,
    neb={
        "type": "cue",
        "all_params": tengri.FIXED,
        "neb_logU": tengri.Fixed(-2.5),
        "neb_logZ_gas": tengri.Fixed(-0.3),
        "neb_fesc": tengri.Fixed(0.0),
        "neb_fesc_lya": tengri.Fixed(0.0),
        "neb_dig_frac": tengri.Fixed(0.0),
    },
    redshift=tengri.Fixed(0.01),
)
model_none = tengri.SEDModel.build(
    ssp,
    sfh=sfh,
    dust=dust,
    neb={"type": "none", "all_params": tengri.FIXED},
    redshift=tengri.Fixed(0.01),
)

p_cue = dict(model_cue.spec.sample(jax.random.PRNGKey(0)))
p_none = {k: v for k, v in p_cue.items() if not k.startswith("neb_")}
out_cue = model_cue.predict(p_cue)
out_none = model_none.predict(p_none)
wave = np.asarray(model_cue.wavelengths)
sed_cue = np.asarray(out_cue.rest_sed())
# neb='none' yields a shorter rest-frame grid (no Cue emission-line wavelengths),
# so interpolate it onto the Cue grid to share the wavelength masks below.
sed_none = np.asarray(out_none.rest_sed(wave))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)

for ax, (lo, hi, lines) in zip(
    axes,
    [
        (4750, 5100, [(4861.33, r"H$\beta$"), (4958.92, r"[O III]"), (5006.84, r"[O III]")]),
        (6400, 6750, [(6548.05, r"[N II]"), (6564.61, r"H$\alpha$"), (6583.45, r"[N II]")]),
    ],
):
    m = (wave > lo) & (wave < hi)
    ax.semilogy(wave[m], sed_none[m], color="0.55", lw=1.4, label="neb = none")
    ax.semilogy(wave[m], sed_cue[m], color="C3", lw=1.4, label="neb = cue")
    for lam, lbl in lines:
        ax.axvline(lam, ls=":", color="0.7", lw=0.8)
        ax.text(lam, ax.get_ylim()[1] * 0.65, " " + lbl, fontsize=8, color="0.4")
    ax.set_xlabel(r"Rest $\lambda$ [$\mathrm{\AA}$]")
    ax.set_ylabel(r"$L_\nu$ [erg s$^{-1}$ Hz$^{-1}$]")
    ax.legend(frameon=False, fontsize=9, loc="upper left")

fig.tight_layout()
plt.savefig("plot_swap_nebular_backend.png", dpi=150, bbox_inches="tight")